In [117]:
import requests
import geopandas as gpd
import json
from shapely.geometry import shape

# URL запроса
url = "https://nspd.gov.ru/api/aeggis/v4/36048/wms"
params = {
    "REQUEST": "GetFeatureInfo",
    "QUERY_LAYERS": "36048",
    "SERVICE": "WMS",
    "VERSION": "1.3.0",
    "FORMAT": "image/png",
    "STYLES": "",
    "TRANSPARENT": "true",
    "LAYERS": "36048",
    "RANDOM": "0.12828180732088468",
    "INFO_FORMAT": "application/json",
    "FEATURE_COUNT": "10",
    "I": "201",
    "J": "316",
    "WIDTH": "512",
    "HEIGHT": "512",
    "CRS": "EPSG:3857",
    "BBOX": "3130860.6785608195,8140237.7642581295,3757032.8142729835,8766409.899970293"
}



# Отправка запроса
response = requests.get(url, params=params, verify=False)
data = response.json()

# Преобразование в GeoDataFrame
features = data.get("features", [])
geoms = [shape(f["geometry"]) for f in features]
attrs = [f["properties"] for f in features]

gdf = gpd.GeoDataFrame(attrs, geometry=geoms, crs="EPSG:3857")

# Преобразуем в WGS84, если нужно
#gdf = gdf.to_crs(epsg=4326)

gdf.head()

,cadastralDistrictsCode,category,descr,externalKey,geom_data_id,label,options,subcategory,system_info,geometry
0,78,36368,78:32:0001676:3999,78:32:0001676:3999,426354188,78:32:0001676:3999,"{'area': None, 'status': 'Учтенный', 'cad_num'...",5,"{'updated': '2025-02-05T17:26:07.668605', 'ins...","POLYGON ((3373004.44 8380148.621, 3373008.658 ..."
1,78,36368,78:32:0001679:3,78:32:0001679:3,155135205,78:32:0001679:3,"{'area': None, 'status': 'Ранее учтенный', 'ca...",5,"{'updated': '2025-02-05T16:16:52.988075', 'ins...","POLYGON ((3374635.7 8380171.432, 3374655.255 8..."
2,78,36368,78:06:0002079:1592,78:06:0002079:1592,155131697,78:06:0002079:1592,"{'area': None, 'status': 'Учтенный', 'cad_num'...",5,"{'updated': '2025-02-05T18:16:41.759469', 'ins...","POLYGON ((3369792.195 8385162.551, 3369749.277..."
3,78,36368,78:06:0002038:4121,78:06:0002038:4121,155131394,78:06:0002038:4121,"{'area': None, 'status': 'Учтенный', 'cad_num'...",5,"{'updated': '2025-02-05T16:16:51.937561', 'ins...","POLYGON ((3369803.403 8387192.435, 3369811.12 ..."
4,78,36368,78:31:0001282:3199,78:31:0001282:3199,135819688,78:31:0001282:3199,"{'area': None, 'status': 'Учтенный', 'cad_num'...",5,"{'updated': '2025-02-05T16:02:05.245654', 'ins...","POLYGON ((3378556.552 8385647.343, 3378591.4 8..."


In [121]:
import requests
import geopandas as gpd
from shapely.geometry import shape
from tqdm import tqdm
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Параметры WMS-запроса
url = "https://nspd.gov.ru/api/aeggis/v4/36048/wms"
width, height = 512, 512

# Bounding Box в EPSG:3857 (ограничивает область, по которой прокликиваем)3380351.1388836354
# bbox = [3130860.6785608195, 8140237.7642581295, 3757032.8142729835, 8766409.899970293]
bbox = [3380351.1388836354, 8384836.254770694, 3385243.1086938865, 8389728.22458094]
# Фиксированные параметры, не меняются при проходе по пикселям
base_params = {
    "REQUEST": "GetFeatureInfo",
    "QUERY_LAYERS": "36048",
    "SERVICE": "WMS",
    "VERSION": "1.3.0",
    "FORMAT": "image/png",
    "STYLES": "",
    "TRANSPARENT": "true",
    "LAYERS": "36048",
    "RANDOM": "0.12828180732088468",
    "INFO_FORMAT": "application/json",
    "FEATURE_COUNT": "10",
    "CRS": "EPSG:3857",
    "WIDTH": str(width),
    "HEIGHT": str(height),
    "BBOX": ",".join(map(str, bbox))
}

# Хранилище результатов
all_geoms = []
all_attrs = []

# Общий прогресс
total_pixels = width * height
print("Начинаем парсинг всех пикселей...")

for j in tqdm(range(height), desc="Строки"):
    for i in range(width):
        params = base_params.copy()
        params["I"] = str(i)
        params["J"] = str(j)

        try:
            response = requests.get(url, params=params, verify=False, timeout=10)
            if response.status_code != 200:
                print(f"Ошибка ответа на пикселе ({i}, {j}): {response.status_code}")
                continue

            data = response.json()
            features = data.get("features", [])

            if not features:
                print(f"Нет данных в пикселе ({i}, {j})")
                continue

            for feature in features:
                geom = shape(feature["geometry"])
                props = feature["properties"]
                all_geoms.append(geom)
                all_attrs.append(props)

        except Exception as e:
            print(f"Ошибка при обработке пикселя ({i}, {j}): {e}")

# Создание GeoDataFrame
if all_geoms:
    gdf = gpd.GeoDataFrame(all_attrs, geometry=all_geoms, crs="EPSG:3857")
    print(f"Получено объектов: {len(gdf)}")
else:
    print("Не удалось получить ни одного объекта.")

Начинаем парсинг всех пикселей...


Строки:   0%|          | 0/512 [00:00<?, ?it/s]

Нет данных в пикселе (117, 0)
Нет данных в пикселе (118, 0)
Нет данных в пикселе (119, 0)
Нет данных в пикселе (120, 0)
Нет данных в пикселе (121, 0)
Нет данных в пикселе (122, 0)
Нет данных в пикселе (123, 0)
Нет данных в пикселе (124, 0)
Нет данных в пикселе (125, 0)
Нет данных в пикселе (126, 0)
Нет данных в пикселе (127, 0)
Нет данных в пикселе (128, 0)
Нет данных в пикселе (129, 0)
Нет данных в пикселе (130, 0)
Нет данных в пикселе (131, 0)
Нет данных в пикселе (132, 0)
Нет данных в пикселе (133, 0)
Нет данных в пикселе (134, 0)
Нет данных в пикселе (135, 0)
Нет данных в пикселе (136, 0)
Нет данных в пикселе (137, 0)
Нет данных в пикселе (138, 0)
Нет данных в пикселе (139, 0)
Нет данных в пикселе (140, 0)
Нет данных в пикселе (141, 0)
Нет данных в пикселе (142, 0)
Нет данных в пикселе (143, 0)
Нет данных в пикселе (144, 0)
Нет данных в пикселе (145, 0)
Нет данных в пикселе (146, 0)
Нет данных в пикселе (147, 0)
Нет данных в пикселе (148, 0)
Нет данных в пикселе (149, 0)
Нет данных

Строки:   0%|          | 1/512 [01:03<9:02:55, 63.75s/it]

Нет данных в пикселе (116, 1)
Нет данных в пикселе (117, 1)
Нет данных в пикселе (118, 1)
Нет данных в пикселе (119, 1)
Нет данных в пикселе (120, 1)
Нет данных в пикселе (121, 1)
Нет данных в пикселе (122, 1)
Нет данных в пикселе (123, 1)
Нет данных в пикселе (124, 1)
Нет данных в пикселе (125, 1)
Нет данных в пикселе (126, 1)
Нет данных в пикселе (127, 1)
Нет данных в пикселе (128, 1)
Нет данных в пикселе (129, 1)
Нет данных в пикселе (130, 1)
Нет данных в пикселе (131, 1)
Нет данных в пикселе (132, 1)
Нет данных в пикселе (133, 1)
Нет данных в пикселе (134, 1)
Нет данных в пикселе (135, 1)
Нет данных в пикселе (136, 1)
Нет данных в пикселе (137, 1)
Нет данных в пикселе (138, 1)
Нет данных в пикселе (139, 1)
Нет данных в пикселе (140, 1)
Нет данных в пикселе (141, 1)
Нет данных в пикселе (142, 1)
Нет данных в пикселе (143, 1)
Нет данных в пикселе (144, 1)
Нет данных в пикселе (145, 1)
Нет данных в пикселе (146, 1)
Нет данных в пикселе (147, 1)
Нет данных в пикселе (148, 1)
Нет данных

Строки:   0%|          | 1/512 [01:27<12:21:49, 87.10s/it]

Нет данных в пикселе (194, 1)


KeyboardInterrupt: 

In [ ]:
# gdf.to_file('west_cadastr.geojson')

In [ ]:
gdf.explore()